# 04. Visualizaciones

Esta notebook transforma los hallazgos del EDA en visualizaciones claras y reutilizables para el portafolio. Se trabaja exclusivamente con la version reducida del dataset para mantener el foco narrativo del proyecto.

## Alcance

- cargar el dataset reducido
- preparar columnas auxiliares para visualizacion
- construir graficos sobre distribucion, experiencia, salario y stacks tecnologicos
- exportar figuras listas para presentacion o documentacion

**Nota:** la notebook usa `matplotlib` puro para evitar dependencias extra y asegurar reproducibilidad en este entorno.


## Subpaso 1. Carga del dataset y configuracion visual

**Proposito:** trabajar con un entorno reproducible y con una carpeta de salida dedicada para las figuras.

**Resultado esperado:** acceso al archivo `data/cleaned_outputs/survey_data_cleaned_reduced.csv` y exportacion a `data/visualization_outputs`.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
INPUT_PATH = DATA_DIR / "cleaned_outputs" / "survey_data_cleaned_reduced.csv"
OUTPUT_DIR = DATA_DIR / "visualization_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_PATH)

plt.style.use("ggplot")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

INPUT_PATH, OUTPUT_DIR


## Subpaso 2. Preparacion de helpers y columnas auxiliares

**Proposito:** dejar listas conversiones utiles para edad, conteos multiseleccion y guardado de figuras.

**Resultado:** una copia de trabajo con `Age_num` y funciones para extraer Top 10 desde columnas multiseleccion.

**Hallazgo operativo:** convertir los grupos etarios a puntos medios facilita graficos de dispersion sin perder la semantica ordinal de la variable original.


In [ ]:
AGE_MID_MAP = {
    "Under 18 years old": 17,
    "18-24 years old": 21,
    "25-34 years old": 29.5,
    "35-44 years old": 39.5,
    "45-54 years old": 49.5,
    "55-64 years old": 59.5,
    "65 years or older": 70,
    "Prefer not to say": None,
}

df_plot = df.copy()
df_plot["Age_num"] = pd.to_numeric(df_plot["Age"].map(AGE_MID_MAP), errors="coerce")

def top_multiselect_counts(series: pd.Series, top_n: int = 10, label: str = "item") -> pd.DataFrame:
    cleaned = series.dropna().astype(str)
    cleaned = cleaned[~cleaned.isin(["", "Not specified"])]
    exploded = cleaned.str.split(";").explode().str.strip()
    return exploded.value_counts().head(top_n).rename_axis(label).reset_index(name="count")

def save_current_figure(filename: str) -> None:
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=300, bbox_inches="tight")

df_plot[["Age", "Age_num"]].head()


## Subpaso 3. Distribucion de compensacion anual convertida

**Proposito:** observar la forma general de la variable salarial principal del proyecto.

**Resultado:** un histograma con recorte visual en el percentil 99 mejora la legibilidad sin ocultar que la distribucion esta sesgada a la derecha.

**Hallazgos:** la compensacion muestra una cola larga y fuerte asimetria positiva; por eso conviene complementar promedios con mediana y comparaciones por grupos.


In [ ]:
salary_vis = df_plot["ConvertedCompYearly"].clip(upper=df_plot["ConvertedCompYearly"].quantile(0.99))

plt.figure(figsize=(10, 5))
plt.hist(salary_vis, bins=30, color="#4C78A8", alpha=0.85)
plt.title("Distribution of Converted Annual Compensation (trimmed at p99)")
plt.xlabel("ConvertedCompYearly")
plt.ylabel("Respondent count")
save_current_figure("viz_01_salary_distribution.png")
plt.show()


## Subpaso 4. Edad y experiencia laboral

**Proposito:** visualizar si el patron positivo entre edad y experiencia visto en EDA tambien aparece de forma clara en un grafico.

**Resultado:** una nube de puntos muestra la relacion entre `Age_num` y `WorkExp`.

**Hallazgos:** aparece una tendencia positiva visible, aunque con dispersion importante dentro de cada grupo etario, lo que sugiere trayectorias profesionales heterogeneas.


In [ ]:
scatter_df = df_plot[["Age_num", "WorkExp"]].dropna().sample(min(3000, len(df_plot)), random_state=42)

plt.figure(figsize=(9, 5))
plt.scatter(scatter_df["Age_num"], scatter_df["WorkExp"], alpha=0.35, color="#59A14F")
plt.title("Age vs Work Experience")
plt.xlabel("Age midpoint")
plt.ylabel("WorkExp")
save_current_figure("viz_02_age_vs_workexp.png")
plt.show()


## Subpaso 5. Distribucion de trabajo remoto

**Proposito:** mostrar de forma simple como se reparte la muestra entre trabajo remoto, hibrido y presencial.

**Resultado:** un grafico de barras resume la composicion del dataset por modalidad de trabajo.

**Hallazgos:** `Remote` y `Hybrid` concentran la mayor parte de respuestas, lo que refuerza que este eje merece comparaciones posteriores de salario y perfil profesional.


In [ ]:
remote_counts = df_plot["RemoteWork"].value_counts().reset_index()
remote_counts.columns = ["RemoteWork", "count"]

plt.figure(figsize=(8, 5))
plt.bar(remote_counts["RemoteWork"], remote_counts["count"], color=["#E15759", "#76B7B2", "#F28E2B"])
plt.title("Remote Work Distribution")
plt.xlabel("")
plt.ylabel("Respondent count")
plt.xticks(rotation=15, ha="right")
save_current_figure("viz_03_remotework_distribution.png")
plt.show()


## Subpaso 6. Compensacion por tipo de empleo

**Proposito:** comparar la distribucion salarial entre las categorias de empleo mas frecuentes.

**Resultado:** un box plot permite ver dispersion, mediana y valores altos dentro de cada categoria.

**Hallazgos:** el empleo full-time domina la muestra, pero las categorias que combinan trabajo independiente o ejecutivo muestran perfiles salariales mas dispersos. Este grafico es util para discutir heterogeneidad, no solo tendencia central.


In [ ]:
employment_order = df_plot["Employment"].value_counts().head(6).index.tolist()
employment_box = df_plot[df_plot["Employment"].isin(employment_order)][["Employment", "ConvertedCompYearly"]].dropna()
employment_groups = [employment_box.loc[employment_box["Employment"] == category, "ConvertedCompYearly"].values for category in employment_order]

plt.figure(figsize=(12, 6))
plt.boxplot(employment_groups, labels=employment_order, patch_artist=True)
plt.title("Converted Annual Compensation by Employment Type")
plt.xlabel("Employment")
plt.ylabel("ConvertedCompYearly")
plt.xticks(rotation=35, ha="right")
save_current_figure("viz_04_salary_by_employment.png")
plt.show()


## Subpaso 7. Compensacion por tipo de desarrollador

**Proposito:** contrastar la distribucion salarial entre los perfiles de desarrollador mas frecuentes.

**Resultado:** el grafico expande la columna multiseleccion `DevType` y compara los cinco roles mas comunes.

**Hallazgos:** `Developer, full-stack` domina por frecuencia, pero la dispersion salarial entre roles sugiere que la especializacion del perfil tambien puede importar al analizar compensacion.


In [ ]:
devtype_counts = top_multiselect_counts(df_plot["DevType"], top_n=5, label="DevType")["DevType"].tolist()
devtype_exploded = (
    df_plot[["DevType", "ConvertedCompYearly"]]
    .dropna(subset=["DevType", "ConvertedCompYearly"])
    .assign(DevType=lambda d: d["DevType"].str.split(";"))
    .explode("DevType")
)
devtype_exploded["DevType"] = devtype_exploded["DevType"].str.strip()
devtype_box = devtype_exploded[devtype_exploded["DevType"].isin(devtype_counts)]
devtype_groups = [devtype_box.loc[devtype_box["DevType"] == category, "ConvertedCompYearly"].values for category in devtype_counts]

plt.figure(figsize=(12, 6))
plt.boxplot(devtype_groups, labels=devtype_counts, patch_artist=True)
plt.title("Converted Annual Compensation for Top Developer Types")
plt.xlabel("Developer type")
plt.ylabel("ConvertedCompYearly")
plt.xticks(rotation=35, ha="right")
save_current_figure("viz_05_salary_by_devtype.png")
plt.show()


## Subpaso 8. Lenguajes actuales vs deseados

**Proposito:** resumir en una sola figura la diferencia entre tecnologias actualmente usadas y tecnologias con interes futuro.

**Resultado:** un grafico de barras doble compara Top 10 actuales y Top 10 deseados para lenguajes.

**Hallazgos:** `JavaScript`, `SQL`, `TypeScript` y `Python` mantienen presencia fuerte en ambos lados. `Go` y `Rust` destacan como tecnologias con interes futuro relativamente alto respecto a su posicion actual.


In [ ]:
current_languages_top10 = top_multiselect_counts(df_plot["LanguageHaveWorkedWith"], label="language")
future_languages_top10 = top_multiselect_counts(df_plot["LanguageWantToWorkWith"], label="language")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].bar(current_languages_top10["language"], current_languages_top10["count"], color="#4C78A8")
axes[0].set_title("Top 10 Programming Languages Used")
axes[0].set_xlabel("")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=35)

axes[1].bar(future_languages_top10["language"], future_languages_top10["count"], color="#59A14F")
axes[1].set_title("Top 10 Programming Languages Desired Next Year")
axes[1].set_xlabel("")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=35)

save_current_figure("viz_06_language_trends.png")
plt.show()


## Subpaso 9. Bases de datos actuales vs deseadas

**Proposito:** visualizar la continuidad y el cambio en las preferencias de bases de datos.

**Resultado:** una figura doble compara uso actual y preferencia futura.

**Hallazgos:** `PostgreSQL` lidera con claridad en ambos escenarios. `Redis` y `Supabase` ayudan a contar una historia de interes creciente por stacks modernos y componentes de alto rendimiento.


In [ ]:
current_databases_top10 = top_multiselect_counts(df_plot["DatabaseHaveWorkedWith"], label="database")
future_databases_top10 = top_multiselect_counts(df_plot["DatabaseWantToWorkWith"], label="database")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].bar(current_databases_top10["database"], current_databases_top10["count"], color="#F28E2B")
axes[0].set_title("Top 10 Databases Used")
axes[0].set_xlabel("")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=35)

axes[1].bar(future_databases_top10["database"], future_databases_top10["count"], color="#E15759")
axes[1].set_title("Top 10 Databases Desired Next Year")
axes[1].set_xlabel("")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=35)

save_current_figure("viz_07_database_trends.png")
plt.show()


## Subpaso 10. Exportacion y cierre de la etapa

**Proposito:** dejar una referencia clara de las figuras producidas en esta notebook.

**Resultado:** las figuras quedan guardadas en `data/visualization_outputs` y listas para reutilizar en README, presentacion o documentacion del caso.

**Hallazgo operativo:** mantener las figuras como artefactos de salida facilita separar exploracion, visualizacion y dashboarding dentro del repositorio curado.


In [ ]:
generated_figures = sorted(path.name for path in OUTPUT_DIR.glob('viz_*.png'))
generated_figures
